# 03 · EEGNet

A compact convolutional network that learns temporal filters, then spatial filters per temporal filter, then a separable convolution, from the 12 × 128 epoch directly (no hand-made features).

*Data: Gao et al. (2014), public Dryad release, doi:10.5061/dryad.2qc64. Own analysis; the plan (folds, training task, decision rule) was fixed before results.*

In [1]:
import os, json, numpy as np, matplotlib.pyplot as plt
# Point GAO_DIR at the "14 electrodes-30 subjects" folder of the Dryad release; GKT_WORK holds all derived files
os.environ.setdefault("GAO_DIR", "./data/14 electrodes-30 subjects/")
os.environ.setdefault("GKT_WORK", "./work/")
from gkt_utils import *
print("data:", GAO_DIR, "| work:", WORK)
import time, torch, torch.nn as nn
torch.set_num_threads(max(1, os.cpu_count() or 1))

data: ./data/14 electrodes-30 subjects/ | work: ./work/


In [2]:
X, G, PID, SES, STIM = load_dataset()
PIDS = sorted(set(PID.tolist()))
print(X.shape, "epochs x channels x samples |", len(PIDS), "people |",
      {k: int((STIM == v).sum()) for k, v in STIM_CODE.items()})

(23388, 12, 128) epochs x channels x samples | 30 people | {'irrelevant': 15583, 'probe': 3890, 'target': 3915}


## Architecture

In [3]:
class EEGNet(nn.Module):
    """Lawhern et al. 2018 style: temporal conv -> depthwise spatial conv -> separable conv -> dense. 1,361 parameters."""
    def __init__(s, C=12, T=128, F1=8, D=2, F2=16, k=64, p=0.5):
        super().__init__()
        s.b1 = nn.Sequential(nn.Conv2d(1, F1, (1, k), padding=(0, k // 2), bias=False), nn.BatchNorm2d(F1),       # 8 temporal filters, 64 samples
                             nn.Conv2d(F1, F1 * D, (C, 1), groups=F1, bias=False), nn.BatchNorm2d(F1 * D),     # 2 spatial filters per temporal filter
                             nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(p))
        s.b2 = nn.Sequential(nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),   # separable: depthwise in time
                             nn.Conv2d(F1 * D, F2, 1, bias=False),                                              #            + pointwise mix
                             nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(p))
        with torch.no_grad(): n = s.b2(s.b1(torch.zeros(1, 1, C, T))).numel()
        s.fc = nn.Linear(n, 1)
    def forward(s, x): return s.fc(s.b2(s.b1(x.unsqueeze(1))).flatten(1)).squeeze(1)   # x: batch x 12 x 128

net = EEGNet()
print(net)
print('trainable parameters:', sum(p.numel() for p in net.parameters() if p.requires_grad))

EEGNet(
  (b1): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): Conv2d(8, 16, kernel_size=(12, 1), stride=(1, 1), groups=8, bias=False)
    (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (4): ELU(alpha=1.0)
    (5): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (6): Dropout(p=0.5, inplace=False)
  )
  (b2): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (fc): Linear(in_features=64, out_features=1, bias

## Training plan
* Same 15 leave-one-pair-out folds; training task **target vs irrelevant**; probes never used in training.
* Channel mean/SD from training epochs only.
* Early stopping on 2 inner validation pairs (never the test pair), patience 5, at most 30 epochs.
* Adam (lr 1e-3), batch 64 × 12 × 128, class-weighted BCE loss, seed 0 in every fold.

`TRAIN = True` retrains every fold (about 30–40 min on a 2-core CPU; faster on a GPU). `TRAIN = False` loads the saved fold weights from `work/weights/` and recomputes the scores, which reproduces the reported results exactly.

In [4]:
TRAIN = not os.path.exists(P("weights/EEGNet_fold1.pt"))   # set True to retrain from scratch
os.makedirs(P("weights"), exist_ok=True)
scores = np.full(len(X), np.nan); log = []; t0 = time.time()
for k, (pl, ph) in enumerate(PAIRS, 1):
    torch.manual_seed(0); np.random.seed(0)
    te = np.isin(PID, [pl, ph]); vp = inner_val(k)
    base = (~te) & (STIM != 1); va = base & np.isin(PID, list(vp)); tr = base & ~np.isin(PID, list(vp))
    mu = X[tr].mean((0, 2), keepdims=True); sd = X[tr].std((0, 2), keepdims=True)
    f = lambda m: torch.tensor((X[m] - mu) / sd, dtype=torch.float32)
    Xtr, Xva, Xte = f(tr), f(va), f(te)
    ytr = torch.tensor((STIM[tr] == 2).astype(np.float32)); yva = torch.tensor((STIM[va] == 2).astype(np.float32))
    lossf = nn.BCEWithLogitsLoss(pos_weight=(ytr == 0).sum() / (ytr == 1).sum())
    net = EEGNet()
    if TRAIN:
        opt = torch.optim.Adam(net.parameters(), lr=1e-3); best = (1e9, None, 0); bad = 0; g = torch.Generator().manual_seed(0)
        for ep in range(30):
            net.train(); perm = torch.randperm(len(Xtr), generator=g)
            for i in range(0, len(perm), 64):
                b = perm[i:i + 64]; opt.zero_grad(); l = lossf(net(Xtr[b]), ytr[b]); l.backward(); opt.step()
            net.eval()
            with torch.no_grad():
                trl = float(np.mean([lossf(net(Xtr[i:i + 2048]), ytr[i:i + 2048]).item() for i in range(0, len(Xtr), 2048)]))
                vl = lossf(net(Xva), yva).item()
            log.append((k, ep, trl, vl))
            if vl < best[0] - 1e-4: best = (vl, {kk: v.clone() for kk, v in net.state_dict().items()}, ep); bad = 0
            else:
                bad += 1
                if bad >= 5: break
        net.load_state_dict(best[1]); torch.save(net.state_dict(), P(f"weights/EEGNet_fold{k}.pt"))
        msg = f"best epoch {best[2]}, val loss {best[0]:.3f}"
    else:
        net.load_state_dict(torch.load(P(f"weights/EEGNet_fold{k}.pt"))); msg = "loaded saved weights"
    net.eval()
    with torch.no_grad(): scores[te] = torch.cat([net(Xte[i:i + 2048]) for i in range(0, len(Xte), 2048)]).numpy()
    print(f"fold {k:2d} ({code(pl)} + {code(ph)}): {msg} | {time.time() - t0:.0f} s")
np.save(P("scores_EEGNet.npy"), scores)

fold  1 (L01 + H01): loaded saved weights | 1 s


fold  2 (L02 + H02): loaded saved weights | 1 s


fold  3 (L03 + H03): loaded saved weights | 2 s


fold  4 (L04 + H04): loaded saved weights | 2 s


fold  5 (L05 + H05): loaded saved weights | 3 s


...

fold 11 (L11 + H11): loaded saved weights | 6 s


fold 12 (L12 + H12): loaded saved weights | 6 s


fold 13 (L13 + H13): loaded saved weights | 7 s


fold 14 (L14 + H14): loaded saved weights | 7 s


fold 15 (L15 + H15): loaded saved weights | 7 s


In [5]:
if log:
    L = np.array(log); fig, ax = plt.subplots(figsize=(7, 3))
    for k in range(1, 16):
        m = L[:, 0] == k; ax.plot(L[m, 1], L[m, 3], color="C1", alpha=.5); ax.plot(L[m, 1], L[m, 2], color="C0", alpha=.5)
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("training (blue) and validation (orange) loss, 15 folds"); plt.show()
else: print("weights loaded: no training curves in this run (set TRAIN = True to see them)")

weights loaded: no training curves in this run (set TRAIN = True to see them)


In [6]:
summ, persons = evaluate(scores, PID, STIM)
show(summ, "EEGNet")
print("errors:", [r["code"] for r in persons if r["decision"] != r["group"]])
json.dump(dict(summary=summ, persons=persons), open(P("results_EEGNet.json"), "w"), indent=1)

EEGNet                   trial AUC 0.908 | person AUC 0.924 (perm p = 0.0001) | detected 14/15 | cleared 13/15 | BA 0.900
errors: ['H08', 'H14', 'L01']


Expected: trial AUC 0.91; 14/15 detected, 13/15 cleared (BA 0.90); errors L01, H08, H14.